# Causal Inference in Practice
## Week 7 — Weighting & Doubly Robust Estimation · Practice Notebook

> **Block II — Adjustment for confounding**
>
> Inverse-probability weighting, stabilized weights, and estimators that survive one modeling mistake.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · A confounded cohort with a known answer

We simulate an epidemiological-style cohort with two covariates `X1, X2` that **confound** a treatment `T` and an outcome `Y`. The confounding is deliberately **nonlinear**, because that is what makes weighting (and double robustness) earn their keep. The **true ATE is exactly 3.0** — we'll keep checking our estimates against it.

Because treatment and outcome share the same nonlinear drivers, the naive difference in means will be badly biased.

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression

TRUE_ATE = 3.0
n = 20_000

def simulate(n):
    X1 = RNG.uniform(-2, 2, n)
    X2 = RNG.uniform(-2, 2, n)
    # nonlinear, strong confounding of TREATMENT
    logit_e = 1.4*np.sin(1.5*X1) + 1.2*(X2**2 - 1.0) - 0.5
    e_true  = 1/(1 + np.exp(-logit_e))
    T = RNG.binomial(1, e_true)
    # nonlinear, strong confounding of OUTCOME; additive effect = TRUE_ATE
    mu0 = 2.0*np.sin(1.5*X1) + 2.0*X2**2 + 1.0*X1
    Y = mu0 + TRUE_ATE*T + RNG.normal(size=n)
    return pd.DataFrame({'X1': X1, 'X2': X2, 'T': T, 'Y': Y}), e_true

df, e_true = simulate(n)
T = df['T'].to_numpy()
Y = df['Y'].to_numpy()
print(f'n = {n},  treated fraction = {T.mean():.3f}')
print(f'true propensity range: [{e_true.min():.3f}, {e_true.max():.3f}]')
print(f'TRUE ATE = {TRUE_ATE}')

### The naive estimate is biased

Compare the treated and control means directly — ignoring that the two groups have very different covariate profiles.

In [ ]:
naive = Y[T == 1].mean() - Y[T == 0].mean()
print(f'naive difference in means = {naive:6.3f}')
print(f'true ATE                  = {TRUE_ATE:6.3f}')
print(f'naive bias                = {naive - TRUE_ATE:+.3f}')
assert abs(naive - TRUE_ATE) > 1.0, 'naive should be badly biased here'

## 2 · The propensity score & the pseudo-population

We fit `e(X) = P(T=1 | X)` with logistic regression. To capture the nonlinear assignment we feed it a **rich basis** (`sin(1.5·X1)`, `X2²`, and the raw covariates) — this is our *correct* propensity model. We'll deliberately break it later.

The inverse-probability weights are `1/e` for the treated and `1/(1−e)` for the controls. In the reweighted **pseudo-population**, treatment no longer tracks the confounders, so a weighted difference in means recovers the ATE.

In [ ]:
def ps_design(d, correct=True):
    """Feature matrix for the propensity model."""
    if correct:
        return np.column_stack([np.sin(1.5*d['X1']), d['X2']**2,
                                d['X1'], d['X2']])
    return np.column_stack([d['X1'], d['X2']])   # misspecified: linear only

def fit_propensity(d, correct=True):
    m = LogisticRegression(max_iter=5000)
    m.fit(ps_design(d, correct), d['T'])
    e = m.predict_proba(ps_design(d, correct))[:, 1]
    return np.clip(e, 1e-3, 1 - 1e-3)   # guard the weights

e = fit_propensity(df, correct=True)
print('estimated propensity range:', f'[{e.min():.3f}, {e.max():.3f}]')

# inverse-probability weights for the ATE
w = np.where(T == 1, 1/e, 1/(1 - e))
print(f'weights: mean = {w.mean():.2f}, max = {w.max():.1f}')

### Estimate the ATE: Horvitz–Thompson vs. Hájek

Two ways to take a weighted difference of means:

- **Horvitz–Thompson** divides each arm's weighted sum by `n` (unbiased, higher variance).
- **Hájek** divides by the *sum of the weights* in that arm (self-normalizing, more stable).

Both should land near the true ATE of 3.0.

In [ ]:
def ipw_ht(weight):
    """Horvitz-Thompson: normalize by n."""
    return (weight*T*Y).mean() - (weight*(1 - T)*Y).mean()

def ipw_hajek(weight):
    """Hajek: normalize each arm by its own weight total."""
    a = np.sum(weight*T*Y)       / np.sum(weight*T)
    b = np.sum(weight*(1 - T)*Y) / np.sum(weight*(1 - T))
    return a - b

ht    = ipw_ht(w)
hajek = ipw_hajek(w)
print(f'Horvitz-Thompson IPW = {ht:6.3f}')
print(f'Hajek IPW            = {hajek:6.3f}')
print(f'true ATE             = {TRUE_ATE:6.3f}')
assert abs(hajek - TRUE_ATE) < 0.25, 'Hajek IPW should recover ~3.0'
assert abs(ht    - TRUE_ATE) < 0.35, 'HT IPW should recover ~3.0'

Weighting recovered the truth while the naive estimate did not. The pseudo-population — the sample reweighted by `w` — has balanced covariates, so the confounding is gone.

### 🔧 Exercise 2.1 — confirm the pseudo-population is balanced

If weighting works, the **weighted** mean of `X2²` should be nearly the same in the treated and control arms, even though the **unweighted** means differ (that imbalance is the confounding). Compute both and compare.

Fill in the `# TODO`s. The skeleton runs as-is (the `...` are placeholders); replace them, then run the solution cell.

In [ ]:
feat = df['X2'].to_numpy()**2

# Unweighted arm means of X2**2 (these should DIFFER -> confounding):
unw_treated = feat[T == 1].mean()
unw_control = feat[T == 0].mean()

# TODO: weighted arm means of X2**2 using w (these should MATCH):
wt_treated = ...   # TODO: np.sum(w*T*feat) / np.sum(w*T)
wt_control = ...   # TODO: weighted control mean

print('unweighted treated/control:', round(unw_treated, 3),
      round(unw_control, 3))
# print('weighted   treated/control:', round(wt_treated, 3),
#       round(wt_control, 3))

### ✅ Solution 2.1

In [ ]:
wt_treated = np.sum(w*T*feat)     / np.sum(w*T)
wt_control = np.sum(w*(1 - T)*feat) / np.sum(w*(1 - T))

print(f'unweighted  treated={unw_treated:.3f}  control={unw_control:.3f}'
      f'  gap={unw_treated - unw_control:+.3f}')
print(f'weighted    treated={wt_treated:.3f}  control={wt_control:.3f}'
      f'  gap={wt_treated - wt_control:+.3f}')
assert abs(unw_treated - unw_control) > 0.2, 'confounding: arms should differ'
assert abs(wt_treated - wt_control) < 0.1, 'weighting should balance X2**2'
print('\nWeighting balanced the covariate -> the pseudo-population works.')

## 3 · Diagnosing & taming the weights

Weighting lives or dies by its **weight distribution**. A few huge weights mean a propensity near 0 or 1 — a near-**positivity** violation — and they inflate the variance. We look at three weightings:

- **raw** unstabilized `1/e`, `1/(1−e)` (mean ≈ 2),
- **stabilized** weights, multiplied by the marginal `P(T=t)` (mean ≈ 1),
- **truncated** weights, capped at the 1st/99th percentile.

In [ ]:
pT = T.mean()
# stabilized weights: multiply by the marginal treatment probability
sw = np.where(T == 1, pT/e, (1 - pT)/(1 - e))

# truncated (raw) weights: cap at the 1st / 99th percentile
lo, hi = np.percentile(w, [1, 99])
wt = np.clip(w, lo, hi)

def eff_n(weight):
    """Kish effective sample size."""
    return weight.sum()**2 / np.sum(weight**2)

for name, ww in [('raw 1/e', w), ('stabilized', sw), ('truncated@99', wt)]:
    print(f'{name:14s}  mean={ww.mean():5.2f}  max={ww.max():6.1f}  '
          f'eff_N={eff_n(ww):8.0f}')

assert sw.max() < w.max(), 'stabilization should shrink the max weight'
assert abs(sw.mean() - 1) < 0.1, 'stabilized weights should average ~1'

In [ ]:
# Visualize the tail-taming. (No plt.show(); the figure is just created.)
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
ax[0].hist(w,  bins=50, color='#2F6DB5'); ax[0].set_title('raw 1/e (mean~2)')
ax[1].hist(sw, bins=50, color='#2A9D8F'); ax[1].set_title('stabilized (mean~1)')
ax[2].hist(wt, bins=50, color='#E9A23B'); ax[2].set_title('truncated @ 99th')
for a in ax:
    a.set_xlabel('weight'); a.set_ylabel('count')
fig.tight_layout()
print('Stabilization shortens the tail; truncation chops it off entirely.')

### Truncation trades bias for variance

The truncated weights are biased — they under-count the rare profiles those large weights represented — but far less variable. Let's see the small bias appear in the point estimate.

In [ ]:
ate_raw   = ipw_hajek(w)
ate_trunc = ipw_hajek(wt)
print(f'IPW (raw weights)        = {ate_raw:6.3f}')
print(f'IPW (truncated @ 99th)   = {ate_trunc:6.3f}')
print(f'true ATE                 = {TRUE_ATE:6.3f}')
print('\nTruncation moved the estimate (a little bias) in exchange for\n'
      'much smaller weights (less variance) -- a deliberate bargain.')

### 🔧 Exercise 3.1 — sweep the truncation threshold

Walk the truncation cap from gentle (99th percentile) to aggressive (80th). As you cap harder, the **bias should grow** (the estimate drifts from 3.0) while the **max weight shrinks** (variance falls). Build the table.

Complete the `# TODO`s below.

In [ ]:
for q in [99, 95, 90, 80]:
    hi_q = np.percentile(w, q)
    w_q  = np.clip(w, None, hi_q)        # cap only the upper tail
    ate_q  = ...     # TODO: ipw_hajek(w_q)
    bias_q = ...     # TODO: ate_q - TRUE_ATE
    # print(f'cap@{q:>2}th  max_w={w_q.max():6.1f}  ATE={ate_q:5.2f} '
    #       f' bias={bias_q:+.2f}')

### ✅ Solution 3.1

In [ ]:
print(f'{"cap":>7} {"max_w":>8} {"ATE":>7} {"bias":>7}')
for q in [99, 95, 90, 80]:
    hi_q = np.percentile(w, q)
    w_q  = np.clip(w, None, hi_q)
    ate_q  = ipw_hajek(w_q)
    bias_q = ate_q - TRUE_ATE
    print(f'{q:>5}th {w_q.max():8.1f} {ate_q:7.2f} {bias_q:+7.2f}')

# Aggressive capping shrinks the max weight but grows the bias.
max_at_99 = np.clip(w, None, np.percentile(w, 99)).max()
max_at_80 = np.clip(w, None, np.percentile(w, 80)).max()
assert max_at_80 < max_at_99, 'capping harder must shrink the max weight'
print('\nbias-variance dial confirmed: tighter cap -> smaller weights, more bias.')

## 4 · AIPW: the doubly robust estimator

Augmented IPW pairs the propensity weights with an **outcome model** `m̂₁(X), m̂₀(X)` (predictions of `Y` under treatment and control). The estimator for each potential-outcome mean is

```
psi1 = m1 + T*(Y - m1)/e          # E[Y(1)]
psi0 = m0 + (1-T)*(Y - m0)/(1-e)  # E[Y(0)]
AIPW = mean(psi1 - psi0)
```

The magic — **double robustness** — is that AIPW is consistent if **either** the propensity model **or** the outcome model is correct. We'll prove it by deliberately breaking one at a time.

In [ ]:
def fit_outcome(d, correct=True):
    """Return predictions m1(X), m0(X) from arm-specific linear models."""
    if correct:
        feats = np.column_stack([np.sin(1.5*d['X1']), d['X2']**2,
                                 d['X1'], d['X2']])
    else:
        feats = np.column_stack([d['X1'], d['X2']])   # misspecified: linear
    tv, yv = d['T'].to_numpy(), d['Y'].to_numpy()
    m1 = LinearRegression().fit(feats[tv == 1], yv[tv == 1]).predict(feats)
    m0 = LinearRegression().fit(feats[tv == 0], yv[tv == 0]).predict(feats)
    return m1, m0

def aipw(e_hat, m1, m0):
    psi1 = m1 + T*(Y - m1)/e_hat
    psi0 = m0 + (1 - T)*(Y - m0)/(1 - e_hat)
    return (psi1 - psi0).mean()

# the 'both correct' baseline
e_good = fit_propensity(df, correct=True)
m1_good, m0_good = fit_outcome(df, correct=True)
print(f'AIPW (both models correct) = {aipw(e_good, m1_good, m0_good):.3f}')
print(f'true ATE                   = {TRUE_ATE:.3f}')

### Break one model at a time — and watch AIPW survive

We now build a **wrong** propensity model and a **wrong** outcome model (both linear-only, blind to the `sin`/square structure). Then we run all four combinations. AIPW should recover ~3.0 in every case **except** when *both* models are wrong.

In [ ]:
e_bad = fit_propensity(df, correct=False)        # wrong PS
m1_bad, m0_bad = fit_outcome(df, correct=False)      # wrong outcome

cases = {
    'both correct      ': aipw(e_good, m1_good, m0_good),
    'PS wrong, OM right': aipw(e_bad,  m1_good, m0_good),
    'PS right, OM wrong': aipw(e_good, m1_bad,  m0_bad),
    'both wrong         ': aipw(e_bad,  m1_bad,  m0_bad),
}
print(f'{"case":20s} {"AIPW":>7} {"|error|":>9}')
for name, val in cases.items():
    print(f'{name:20s} {val:7.3f} {abs(val - TRUE_ATE):9.3f}')

# Double robustness: one wrong model is fine; both wrong is not.
assert abs(cases['PS wrong, OM right'] - TRUE_ATE) < 0.25, 'OM should rescue'
assert abs(cases['PS right, OM wrong'] - TRUE_ATE) < 0.25, 'PS should rescue'
assert abs(cases['both wrong         '] - TRUE_ATE) > 0.8, 'both wrong -> biased'
print('\nDouble robustness confirmed: AIPW survives ONE modeling mistake.')

### Why plain IPW is *not* doubly robust

For contrast: plain IPW relies on the propensity model alone. With the **wrong** propensity model it has no outcome model to fall back on, so it is biased — exactly the situation AIPW rescues.

In [ ]:
ipw_wrong = ipw_hajek(np.where(T == 1, 1/e_bad, 1/(1 - e_bad)))
aipw_rescued = cases['PS wrong, OM right']
print(f'plain IPW, WRONG propensity        = {ipw_wrong:6.3f}  (biased)')
print(f'AIPW, same wrong PS + right outcome = {aipw_rescued:6.3f}  (recovered)')
print(f'true ATE                            = {TRUE_ATE:6.3f}')
assert abs(ipw_wrong - TRUE_ATE) > 0.8, 'plain IPW with wrong PS is biased'
assert abs(aipw_rescued - TRUE_ATE) < 0.25, 'AIPW recovers it'

### 🔧 Exercise 4.1 — AIPW collapses to its pieces

A nice sanity check: if you set the outcome predictions to **zero** (`m1 = m0 = 0`), the AIPW formula reduces to the Horvitz–Thompson IPW estimator. Verify it numerically with the good propensity scores.

Complete the `# TODO`.

In [ ]:
zeros = np.zeros(n)
# TODO: AIPW with zero outcome predictions should equal HT-IPW.
aipw_zero = ...   # TODO: aipw(e_good, zeros, zeros)
ht_ref    = ipw_ht(np.where(T == 1, 1/e_good, 1/(1 - e_good)))
# print(aipw_zero, ht_ref)

### ✅ Solution 4.1

In [ ]:
aipw_zero = aipw(e_good, zeros, zeros)
ht_ref    = ipw_ht(np.where(T == 1, 1/e_good, 1/(1 - e_good)))
print(f'AIPW with zero outcome model = {aipw_zero:.4f}')
print(f'Horvitz-Thompson IPW         = {ht_ref:.4f}')
assert abs(aipw_zero - ht_ref) < 1e-8, 'AIPW with m=0 must equal HT-IPW'
print('Confirmed: AIPW = outcome model + IPW correction. '
      'Zero out the model and only the IPW piece remains.')

## 5 · Wrap-up & self-check

- **IPW** weights each unit by `1 / P(observed treatment | X)`, building a **pseudo-population** in which treatment is unconfounded. The naive estimate was badly biased; IPW recovered the true ATE of 3.0.
- **Horvitz–Thompson** (divide by `n`) is unbiased but noisier; **Hájek** (self-normalizing) is the stable default.
- **Extreme weights** signal near-**positivity** violations. **Stabilization** gives mean-one weights; **truncation** trades a little bias for much less variance.
- **AIPW is doubly robust**: it recovered 3.0 whenever the propensity model **or** the outcome model was right, and only failed when **both** were wrong.

**You're ready for Week 8** if you can explain why weighting by 1/e removes confounding, name the two assumptions it needs, and state the doubly robust property in one sentence. Next week: instrumental variables and Mendelian randomization, where we finally drop the no-unmeasured-confounding assumption.